# Expanded trading-research evidence

This notebook is a read-only, executable showcase of repository components and completed immutable evidence. It does not duplicate training logic or select models. Its canonical evidence source is run `20260928T005423Z-0634efaf`, report `reports/expanded_closeout/20260928T005423Z-0634efaf`, and source revision `e9b3e0cdf57c63c87a9dc881d58b1696f0472ba5`. The loader and the explicit checks below verify the report manifest and every declared table hash before displaying evidence. The final holdout remains descriptive and the expanded close-out remains exploratory until a genuinely untouched confirmation period exists.

## Local execution and future public bootstrap

**Local verified mode (available now).** Run the notebook from this repository checkout after installing `.[phase2,notebook]`. Unless `TRADING_REPORT_DIR` is set, it loads the canonical local report named above and fails if that exact report is unavailable or inconsistent.

**Public clean-runtime mode (not published yet).** The repository remote is `https://github.com/crowleymr/event_based_ml_trading_algo.git`, but no submission tag, release asset, release URL, or public-access verification has been recorded. The next code cell deliberately contains unmistakable publication placeholders. Setting `TRADING_PUBLIC_BOOTSTRAP=1` fails before any download until all placeholders are replaced with an exact tag, immutable release URL, and SHA-256. Do not treat the remote URL alone as evidence that the repository or notebook is public.

At publication, the clean-runtime sequence is: clone the exact repository URL; check out the exact submission tag; install the release lock; download the declared report archive; verify its archive SHA-256; extract it; set `TRADING_REPORT_DIR`; then run all cells. These actions must be tested in a new hosted runtime before the notebook URL is placed in the submission PDF.

In [ ]:
import os

PUBLIC_REPOSITORY_URL = 'https://github.com/crowleymr/event_based_ml_trading_algo.git'
PUBLIC_SUBMISSION_TAG = 'PUBLIC_SUBMISSION_TAG_NOT_PUBLISHED'
PUBLIC_REPORT_ARCHIVE_URL = 'PUBLIC_REPORT_ARCHIVE_URL_NOT_PUBLISHED'
PUBLIC_REPORT_ARCHIVE_SHA256 = 'PUBLIC_REPORT_ARCHIVE_SHA256_NOT_PUBLISHED'

if os.environ.get('TRADING_PUBLIC_BOOTSTRAP') == '1':
    unresolved = [name for name, value in {
        'PUBLIC_SUBMISSION_TAG': PUBLIC_SUBMISSION_TAG,
        'PUBLIC_REPORT_ARCHIVE_URL': PUBLIC_REPORT_ARCHIVE_URL,
        'PUBLIC_REPORT_ARCHIVE_SHA256': PUBLIC_REPORT_ARCHIVE_SHA256,
    }.items() if value.endswith('NOT_PUBLISHED')]
    if unresolved:
        raise RuntimeError('Public bootstrap is not published; unresolved fields: ' + ', '.join(unresolved))


## End-to-end implementation map

- [Configuration and authoritative runner](../src/trading_pipeline/run.py)
- [Market and SEC extraction](../src/trading_pipeline/data/pipeline.py)
- [Security master](../src/trading_pipeline/data/security_master.py)
- [Point-in-time feature construction](../src/trading_pipeline/features/build.py)
- [Model contracts and registry](../src/trading_pipeline/experiments/contracts.py)
- [Temporal optimisation](../src/trading_pipeline/optimisation/splits.py)
- [Supervised model families](../src/trading_pipeline/modelling/supervised_models)
- [RL policies and environment](../src/trading_pipeline/rl)
- [Portfolio execution and costs](../src/trading_pipeline/portfolio/backtest.py)
- [Risk-return frontier evidence](../src/trading_pipeline/portfolio/frontier.py)
- [Generated reporting](../src/trading_pipeline/reporting/generate.py)
- [Read-only dashboard](../src/trading_pipeline/dashboard/app.py)

## Problem, methodology and core concepts

The research question is whether models using daily market information and point-in-time SEC fundamentals can improve five-session cross-sectional stock ranking, and whether that improvement survives portfolio construction and transaction costs. The workflow deliberately separates prediction, portfolio allocation and economic evaluation so that a forecasting improvement is not mistaken for a profitable strategy.

**Point-in-time information.** A historical decision may use only information public at that time. SEC facts become eligible only when the market session is strictly after the filing date. Rolling market features use observations through signal date T, and the implemented backtest enters at the next session's close, T+1. The five-session close-to-close target and tradable T+1 holding window are not identical.

**Temporal validation.** Training fits parameters; chronological validation chooses among declared configurations; outer walk-forward evaluation estimates the complete procedure; and the completed Slice 1 final holdout is descriptive after its selection lock. It must not be used to choose, tune, or reject features, models, portfolio engines, or parameters. Purging removes samples whose five-session label overlaps a later split, while embargo withholds additional boundary sessions. A new selection study needs a new holdout/data vintage and documented protocol.

**Forecast metrics.** MAE and RMSE are errors, so lower is better. Daily information coefficient (IC) is the Spearman rank correlation between predicted and realised returns across stocks, so higher is better. IC is especially relevant because the portfolio selects the highest-ranked securities, but useful ranking does not guarantee useful after-cost returns.

**Portfolio metrics.** Annualised return measures compounded growth on a 252-session convention; higher is generally preferable subject to risk and uncertainty. Annualised volatility measures return variability; lower is less variable, not necessarily better in isolation. The Sharpe ratio divides annualised mean return by annualised volatility under the stated risk-free-rate convention; higher means more return per unit of variability, but it is sample-dependent and does not capture every tail risk. Maximum drawdown is the worst observed peak-to-trough decline; a smaller decline is generally preferable. Turnover measures traded value relative to equity and connects model instability to transaction costs.

**Benchmark comparison.** Beta measures co-movement with SPY. Tracking error measures variability of active return, and the information ratio compares active return with that tracking error. These help distinguish genuine benchmark-relative behaviour from ordinary market exposure.

**Risk views.** A model-conditioned efficient frontier uses a supervised model's expected returns together with a common covariance estimate and constraints. It is not universal. A realised risk-return curve instead connects the conservative, balanced and aggressive backtest outcomes for a model. DQN and PPO belong in the realised comparison because their sleeve-selector policies do not emit a stock-level expected-return vector.

The notebook displays only values loaded from the selected report's generated Parquet tables. The report loader verifies declared output hashes where available. It does not rebuild metrics or infer absent expanded-study evidence. See the [full glossary and methodological cautions](../docs/GLOSSARY.md) for definitions used throughout the report.

In [ ]:
from pathlib import Path
import hashlib
import json
import polars as pl
try:
    from IPython.display import display, Markdown
except ImportError:  # Script-mode validation without a notebook runtime
    def display(value): return value
    def Markdown(value): return value
from trading_pipeline.dashboard import load_expanded_report

CANONICAL_RUN_ID = '20260928T005423Z-0634efaf'
CANONICAL_REPORT_RELATIVE = Path('reports/expanded_closeout') / CANONICAL_RUN_ID
CANONICAL_CODE_VERSION = 'e9b3e0cdf57c63c87a9dc881d58b1696f0472ba5'
CANONICAL_MANIFEST_SHA256 = '4144c9d1bd0c1637eb83021edd7d5359255aa56e73e13e49c5cd1f1d5fd1190a'
EXPECTED_REPORT_TABLES = {
    'pipeline_stage_summary', 'pipeline_security_summary', 'architecture_trial_summary',
    'hpo_trial_summary', 'risk_scenario_summary', 'model_conditioned_frontier_points',
    'model_conditioned_frontier_weights', 'realised_risk_return_curve',
    'final_testbench_metrics', 'final_testbench_equity_curve',
    'question_and_assumption_register',
}

def repository_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository checkout.')

root = repository_root()
configured = os.environ.get('TRADING_REPORT_DIR')
report_dir = Path(configured).expanduser().resolve() if configured else (root / CANONICAL_REPORT_RELATIVE).resolve()
manifest_path = report_dir / 'research_evidence_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Canonical expanded report is unavailable: {manifest_path}')
manifest_digest = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
allow_noncanonical = os.environ.get('TRADING_ALLOW_NONCANONICAL_REPORT') == '1'
if not allow_noncanonical and manifest_digest != CANONICAL_MANIFEST_SHA256:
    raise ValueError(f'Canonical report manifest SHA-256 mismatch: {manifest_digest}')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if not allow_noncanonical and manifest.get('source_run_id') != CANONICAL_RUN_ID:
    raise ValueError(f"Unexpected source run: {manifest.get('source_run_id')}")
if not allow_noncanonical and manifest.get('code_version') != CANONICAL_CODE_VERSION:
    raise ValueError(f"Unexpected source revision: {manifest.get('code_version')}")
declared_tables = {Path(item['relative_path']).stem for item in manifest.get('outputs', [])}
if declared_tables != EXPECTED_REPORT_TABLES:
    raise ValueError(f'Report table contract mismatch: {sorted(declared_tables)}')
loaded = load_expanded_report(report_dir)  # rechecks schemas and every declared output hash
provenance, tables = loaded['provenance'], loaded['tables']
expanded_report = True
report_label = report_dir.relative_to(root) if report_dir.is_relative_to(root) else report_dir
display(Markdown(f"**Source run:** `{provenance.get('source_run_id')}`  \\n**Source revision:** `{provenance.get('code_version')}`  \\n**Report:** `{report_label}`  \\n**Manifest SHA-256:** `{manifest_digest}`  \\n**Verified report tables:** `{len(tables)}`  \\n**Research status:** `{provenance.get('claim_status')}`"))

## Bounded authoritative demonstration (opt-in)

The short demonstration is operational evidence, not the source of the full-study results shown below. It may run only through `python -m trading_pipeline.run --config ...`; the notebook contains no duplicate training path. No licence-compatible public real-data bundle/config has been published yet, so the cell is disabled by default and fails clearly if enabled without an explicit config. `configs/smoke.yaml` is synthetic software verification and must not be described as real-data research evidence.

After a bounded public bundle is approved and published, set `TRADING_RUN_BOUNDED_DEMO=1` and `TRADING_BOUNDED_DEMO_CONFIG` to its repository-relative config. The config and bundle must retain point-in-time joins, chronological purge/embargo, T+1 execution and costs. Full-study tables remain loaded from the immutable canonical report rather than the demonstration run.

In [ ]:
import subprocess
import sys

if os.environ.get('TRADING_RUN_BOUNDED_DEMO') == '1':
    configured_demo = os.environ.get('TRADING_BOUNDED_DEMO_CONFIG')
    if not configured_demo:
        raise RuntimeError('Bounded demo requested, but TRADING_BOUNDED_DEMO_CONFIG is not set.')
    demo_config = (root / configured_demo).resolve()
    if not demo_config.is_relative_to(root) or not demo_config.is_file():
        raise FileNotFoundError(f'Bounded demo config is unavailable inside the repository: {demo_config}')
    subprocess.run([sys.executable, '-m', 'trading_pipeline.run', '--config', str(demo_config)], cwd=root, check=True)
else:
    display(Markdown('*Bounded demonstration not executed; canonical evidence loading continues below.*'))


## Data extraction, feature store and temporal splits

The report provenance identifies its source run and generated inputs. Tables below are generated report products; they are not recomputed in this notebook. Optional expanded-study tables display only when present. The linked implementation map points to repository code, while the selected run/report records archived execution evidence.

In [ ]:
for name in ('pipeline_stage_summary', 'universe_summary', 'split_profile', 'feature_summary'):
    if name in tables:
        display(Markdown(f'### {name.replace("_", " " ).title()}'))
        display(tables[name])

## Architecture selection, HPO and training

When trial summaries are present, compare candidates using their declared objective and metric direction (for example, lower RMSE or higher IC where those are declared). A leaderboard is descriptive evidence of the declared selection protocol; do not rank models using the final holdout. Missing trial tables mean this report does not provide that evidence.

In [ ]:
for name in ('architecture_trial_summary', 'hpo_trial_summary', 'training_summary', 'training_trace'):
    if name in tables:
        frame = tables[name]
        display(Markdown(f'### {name.replace("_", " " ).title()}'))
        display(frame)

## Risk scenarios and frontiers

Model-conditioned efficient frontiers apply only to supervised predictors that emit security-level expected returns. Realised risk-return curves are the common cross-family comparison and include DQN/PPO.

In [ ]:
import matplotlib.pyplot as plt

frontier = tables.get('model_conditioned_frontier_points')
if frontier is not None and not frontier.is_empty():
    latest = frontier['rebalance_date'].max()
    data = frontier.filter((pl.col('rebalance_date') == latest) & (pl.col('status') == 'complete'))
    fig, ax = plt.subplots()
    for model in data['model_id'].unique().sort().to_list():
        series = data.filter(pl.col('model_id') == model).sort('expected_volatility')
        ax.plot(series['expected_volatility'], series['expected_return'], marker='o', label=model)
    ax.set(xlabel='Expected annualised volatility', ylabel='Expected annualised return', title=f'Model-conditioned frontiers: {latest}')
    ax.legend(); plt.show()
else:
    display(Markdown('*No completed model-conditioned frontier evidence is available.*'))

realised = tables.get('realised_risk_return_curve')
if realised is not None and not realised.is_empty():
    fig, ax = plt.subplots()
    groups = realised.select(['model_id', 'seed']).unique().iter_rows() if 'seed' in realised.columns else ((model, None) for model in realised['model_id'].unique().sort().to_list())
    for model, seed in groups:
        series = realised.filter((pl.col('model_id') == model) & (pl.col('seed') == seed) if seed is not None else pl.col('model_id') == model).sort('risk_order')
        label = f'{model} · seed {seed}' if seed is not None else model
        style = '-' if series['risk_control_monotonic'].all() else 'None'
        ax.plot(series['annualised_volatility'], series['annualised_return'], linestyle=style, marker='o', label=label)
        for row in series.iter_rows(named=True):
            ax.annotate(row['risk_scenario'], (row['annualised_volatility'], row['annualised_return']))
    ax.set(xlabel='Realised annualised volatility', ylabel='Realised annualised return', title='Realised model risk-return curves')
    ax.legend(); plt.show()
    display(realised)
else:
    display(Markdown('*No completed three-scenario realised risk-return evidence is available.*'))

## Final test bench and market comparison

In [ ]:
has_final_bench = 'final_testbench_metrics' in tables
comparison = tables['final_testbench_metrics'] if has_final_bench else tables['experiment_comparison']
display(comparison)
equity_source = tables['final_testbench_equity_curve'] if has_final_bench else tables['equity_drawdown_series']
equity = equity_source.filter(pl.col('split').is_in(['test', 'descriptive_holdout']))
fig, ax = plt.subplots()
groups = equity.select(['experiment_id', 'seed', 'risk_scenario']).unique().iter_rows() if 'seed' in equity.columns else ((experiment, None, None) for experiment in equity['experiment_id'].unique().sort().to_list())
for experiment, seed, scenario in groups:
    cell = ((pl.col('experiment_id') == experiment) & (pl.col('seed') == seed) & (pl.col('risk_scenario') == scenario)) if seed is not None else (pl.col('experiment_id') == experiment)
    series = equity.filter(cell).sort('session_date')
    label = f'{experiment} · {scenario} · seed {seed}' if seed is not None else experiment
    ax.plot(series['session_date'], series['equity'], label=label)
ax.set(xlabel='Session', ylabel='Equity (initial = 1)', title='Final test-bench performance against the market')
ax.legend(ncol=2); plt.show()

## Individual-security contribution

In [ ]:
security = tables['pipeline_security_summary'] if 'pipeline_security_summary' in tables else tables['security_contribution_summary']
ticker = os.environ.get('TRADING_TICKER')
if ticker and 'ticker' in security.columns:
    security = security.filter(pl.col('ticker') == ticker.upper())
display(security)

## Optional local dashboard

From the repository root, launch the read-only Streamlit app for the newest completed expanded report:

```powershell
.venv/Scripts/python -m trading_pipeline.dashboard
```

Use `--expanded-report "$env:TRADING_REPORT_DIR"` to pin a particular expanded report, or `--report` for a Slice 1 report. To launch from this notebook, set `LAUNCH_TRADING_DASHBOARD=1`; the app detects the report type and does not train or modify source artefacts. The inline tables and plots above remain the evidence view when Streamlit is unavailable.

In [ ]:
# Set LAUNCH_TRADING_DASHBOARD=1 for an interactive local session.
# The default remains non-blocking so clean notebook execution can finish.
if os.environ.get('LAUNCH_TRADING_DASHBOARD') == '1':
    import subprocess, sys
    dashboard = subprocess.Popen([sys.executable, '-m', 'trading_pipeline.dashboard', '--expanded-report' if expanded_report else '--report', str(report_dir)])
    display(Markdown('Dashboard started at the Streamlit URL printed by the subprocess.'))